# 1. From `linalg` to a binary, one step at a time

The kernel is `vadd`: an element-wise `a + b` over about a million 16-bit
integers, spread over 32 banks of DDR4. It is the smallest kernel that goes
through every stage while staying readable.

Each cell below runs one step and prints part of what it wrote. The files are in
`tutorial/artifacts/`, named `vadd_1048576_ddr_32_<step>`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # tutorial/, for _infra
from _infra import pud

NAME, BANKS, MEM = "vadd_1048576", 32, "ddr"
TAG = f"{NAME}_{MEM}_{BANKS}"
pud.show(pud.kernel(NAME))

Plain `linalg` on ranked tensors: nothing in it is specific to processing in
memory. That is the input a compiler gets.

## Step 1 — to bit-sliced operations (`--convert-linalg-to-bits`)

In [ ]:
bits = pud.to_bits(pud.kernel(NAME), TAG)
pud.show(bits, grep=r"scf.for|bits\.|tensor.empty")

The million-element vector has been cut into **128 tiles of 8192 elements**, and
each tile became a `!bits.slice<16x8192>`: 16 DRAM rows, row *i* holding **bit
*i*** of all 8192 values. That transposed layout is what lets one DRAM command act
on 8192 values at once.

`bits.add_i` is still a single operation on a whole slice. The frontend has chosen
the tiling and the layout, but nothing yet about banks or commands.

## Step 2 — the mapping (`--bits-opt-mapping`)

An ILP decides how the 128 tiles spread over the banks. It is priced with the
command count of every operation, so it can weigh spreading work out against the
copies that spreading costs.

In [ ]:
m = pud.map_banks(bits, TAG, banks=BANKS, mem=MEM)
print(f"model: {m.model}")
print(f"objective (the schedule's cost): {m.objective}   solved in {m.seconds:.1f} s\n")
for loop in pud.mapping_attrs(m.path):
    for k, v in loop.items():
        print(f"  {k:18} {v}")

Read it as: the 128-iteration loop is **parallel**; it is spread over
`parallel_factor` banks; each bank runs `iters_per_bank` tiles, one after the
other; and there is `concurrent_groups` such group of banks. 32 × 4 = 128, the
whole loop. The IR itself is unchanged: the pass only annotates it.

> **Try it:** set `BANKS = 8` or `BANKS = 128` in the first cell and run the
> notebook again. `parallel_factor` and `iters_per_bank` move; the product stays 128.

## Step 3 — one row at a time (`--bits-pattern-apply`)

In [ ]:
rows = pud.expand_rows(m.path, TAG)
pud.show(rows, grep=r"scf.for|row_add|extract_row|insert_row|create_all_zero", lines=20)

The single slice-level addition became **a loop over the 16 bit positions**,
carrying a carry row from one bit to the next: a ripple-carry adder written as a
loop. Every value is now exactly one DRAM row (`!bits.row<8192>`), and
`bits.carry = "resident"` records that the carry stays in a row of its own.

## Step 4 — DRAM commands (`--convert-bits-rows-to-pud`)

Every row operation becomes the AAP/AP sequence that computes it on SIMDRAM's
substrate. The sequences come from **lime**, a logic synthesiser that searches for
the shortest one, and a few are hand-written. The compiler checks every sequence
against a model of the substrate before using it. This is the one for a full adder:

In [ ]:
out = pud.run([pud.rowop_sanity()]).stdout
block = out[out.index("row_add:"):]
print(block[:block.index("\n\n")])

`T0`…`T3` are the four compute rows the substrate reserves; `DCC0`/`DCC1` are the
dual-contact cells and `DCC0N`/`DCC1N` their complements; names like `T0_T1_T2`
are the row *groups* whose activation computes a majority. `in0`, `in1` and
`out0` are data rows. Eight commands per bit, plus one to clear the carry:
`8n + 1` for an *n*-bit add.

In [ ]:
pud_ir = pud.to_pud(rows, TAG)
pud.show(pud_ir, grep=r"scf.for|pud\.AAP|pud\.AP ", lines=24)
print()
for k, v in pud.row_space(pud_ir).items():
    print(f"  pud.{k:20} {v}")

The loop nest now spells the mapping out: an outer loop per group, one per tile a
bank runs, one per bit position, and the **banks innermost**, because all banks
receive the same command and issuing it is a loop over banks.

The module also publishes the **shape of the row space**: how many data rows each
bank needs (at most 494 fit: a subarray has 512 rows and 18 are reserved), the row
stride and the channel size. The runtime lays memory out from exactly these
numbers (notebook 02).

## Step 5 — addresses and instructions (`--convert-pud-rows-to-func`)

In [ ]:
llvm_mlir = pud.to_llvm_dialect(pud_ir, TAG)
pud.show(llvm_mlir, grep=r"pud_arena|inline_asm", lines=6)

A row is now an address: `arena + channel·channel_size + row·(stride·banks_per_channel)
+ bank·stride`. A DRAM command is an **inline instruction**: `0x0F 0x04` followed by
a selector, `.word 97` = 0x61 = AAP, with the rows in `rdi` and `rsi`. It is
marked as having side effects and clobbering memory, so LLVM can neither drop nor
reorder the schedule. The other passes on the command line are MLIR's standard way
out of its dialects into the LLVM dialect.

## Step 6 — LLVM to an object file

In [ ]:
obj = pud.to_object(llvm_mlir, TAG, BANKS, MEM)
pud.show_code(pud.first_row_op_loop(obj), "gas")

That is the first loop in the program that issues a command to every bank, and
every innermost loop has this shape: an address computed from the other one, the
row op, a counter increment, an add of `0x2000` — the 8 KiB row stride, which
moves to the next bank — and the compare-and-branch over the 32 banks. (A plain `objdump -d` does not know the
`0f 04` instruction and garbles the next few lines; the helper prints the row op
from its bytes and disassembles the rest again from the byte after it.) **Six
instructions per DRAM command**; notebook 04 shows what that costs.

Continue with **02_vadd_run** to link this object and run it.